# Neurological MRI XAI Pipeline — Colab Runner

Master orchestrator for training Swin + LoRA, SAM ROI, and Florence-2 reporting.

**Before running:** Set your repo URL in cell 2 and Kaggle credentials in Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) if using Kaggle data source.

In [ ]:
# Cell 1: Environment check
import sys
import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Clone repository
import os
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USERNAME/neuro-mri-xai.git"  # <-- UPDATE THIS
PROJECT_DIR = Path("/content/neuro-mri-xai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"Repo already exists at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 3: Install dependencies
import sys
from pathlib import Path

import torch

# Avoid reinstalling torch on Colab to prevent CUDA mismatch
print(f"Colab torch version: {torch.__version__}")

!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

sys.path.insert(0, str(PROJECT_DIR / "src"))
print("Dependencies installed.")

In [ ]:
# Cell 4: Secrets / authentication
import os
from pathlib import Path

DATA_SOURCE = "kaggle"  # "kaggle" or "gdrive"

if DATA_SOURCE == "kaggle":
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(exist_ok=True)
    (kaggle_dir / "kaggle.json").write_text(
        f'{{"username":"{os.environ["KAGGLE_USERNAME"]}","key":"{os.environ["KAGGLE_KEY"]}}}'
    )
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials configured.")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")

In [ ]:
# Cell 5: Download dataset
!python scripts/download_data.py --source {DATA_SOURCE}

In [ ]:
# Cell 6: Download SAM weights
!python scripts/download_weights.py

In [ ]:
# Cell 7: Config override (optional)
import os

os.environ["NEURO_MRI_DATA_DIR"] = "/content/data"
os.environ["NEURO_MRI_PROJECT_ROOT"] = "/content/neuro-mri-xai"

# Toggle features without editing YAML:
# os.environ["NEURO_MRI_USE_LORA"] = "true"
# os.environ["NEURO_MRI_SAM_ENABLED"] = "false"

from neuro_mri_xai.config import load_config
cfg = load_config("configs/default.yaml")
print(f"Data dir: {cfg.dataset.data_dir}")
print(f"Backbone: {cfg.model.backbone}, LoRA: {cfg.model.use_lora}, SAM: {cfg.sam.enabled}")

In [ ]:
# Cell 8: Train Swin + LoRA
!python -m neuro_mri_xai.train --config configs/default.yaml

In [ ]:
# Cell 9: Evaluate on test set
!python -m neuro_mri_xai.evaluate --config configs/default.yaml --checkpoint outputs/checkpoints/best_swin.pt

In [ ]:
# Cell 10: Explain + Report on a sample image
from pathlib import Path
from IPython.display import HTML, display
from neuro_mri_xai.report import generate_report

data_dir = Path("/content/data")
sample_image = next(data_dir.rglob("*.jpg"))
print(f"Sample: {sample_image}")

report_path = generate_report(
    checkpoint="outputs/checkpoints/best_swin.pt",
    image=str(sample_image),
    config_path="configs/default.yaml",
)
display(HTML(report_path.read_text()))

In [ ]:
# Cell 11: Save outputs to Google Drive
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/neuro-mri-xai-outputs")
dest.mkdir(parents=True, exist_ok=True)

for folder in ["checkpoints", "figures", "reports", "logs"]:
    src = Path("outputs") / folder
    if src.exists():
        shutil.copytree(src, dest / folder, dirs_exist_ok=True)
        print(f"Copied {src} -> {dest / folder}")